In [1]:
import sagemaker

import boto3
from botocore.exceptions import ClientError
import ast

# secret shhhhh
def get_secret():

    secret_name = "gitlab/tcwyu-respository"
    region_name = "us-east-2"

    # Create a Secrets Manager client
    session = boto3.session.Session()
    client = session.client(
        service_name='secretsmanager',
        region_name=region_name
    )

    try:
        get_secret_value_response = client.get_secret_value(
            SecretId=secret_name
        )
    except ClientError as e:
        # For a list of exceptions thrown, see
        # https://docs.aws.amazon.com/secretsmanager/latest/apireference/API_GetSecretValue.html
        raise e

    # Decrypts secret using the associated KMS key.
    secret = get_secret_value_response['SecretString']

    # Your code goes here.
    return secret
    
secret = ast.literal_eval(get_secret())


In [2]:
# git config
git_config = {
    'repo': 'https://git.uwaterloo.ca/medical-ai-lab/bertha_experiments.git',
    'username': secret['username'],
    'password': secret['password']
}

hyperparameters = {
    'epochs': 1, 
    'batch_size': 64, 
    'lr': 1e-4,
}

estimator_args = {
    "base_job_name": "smile-uw-testing-pretraining",
    "role": "arn:aws:iam::776029861160:role/service-role/AmazonSageMaker-ExecutionRole-20230328T101238",
    "output_path": "s3://himss2023mimiciv/uw_skynet/output",
    # "model_uri": "s3://himss2023mimiciv/uw_skynet/models/no-vsa-checkpoint-41000/",
    "image_uri": "776029861160.dkr.ecr.us-east-2.amazonaws.com/smile-uw-containers:medair-huggingface-pytorch-training-test",
    #"checkpoint_s3_uri": "s3://himss2023mimiciv/uw_skynet/output/checkpoints",
    "instance_count": 1,
    "instance_type": "ml.g4dn.xlarge",
    "entry_point": "BERTHA_AWS_Pretraining.py",
    "git_config": git_config,
    "source_dir": '.',
    "hyperparameters": hyperparameters,
}

pytorch_estimator = sagemaker.estimator.Estimator(**estimator_args)

In [3]:
input_data = {
    "train": "s3://himss2023mimiciv/uw_skynet/data/mimic-iv_data/hf_dataset_pretrain/train",
    "test": "s3://himss2023mimiciv/uw_skynet/data/mimic-iv_data/hf_dataset_pretrain/test",
    "tokenizer": "s3://himss2023mimiciv/uw_skynet/data/tokenizer-mimic-iv-icd-final"
}

In [4]:
pytorch_estimator.fit(inputs=input_data)

Cloning into '/tmp/tmptit1jxy3'...
INFO:sagemaker:Creating training-job with name: smile-uw-testing-2023-04-05-22-11-58-199


2023-04-05 22:11:59 Starting - Starting the training job...
2023-04-05 22:12:17 Starting - Preparing the instances for training......
2023-04-05 22:13:19 Downloading - Downloading input data...
2023-04-05 22:14:00 Training - Downloading the training image........................
2023-04-05 22:17:51 Training - Training image download completed. Training in progress..bash: cannot set terminal process group (-1): Inappropriate ioctl for device
bash: no job control in this shell
sed: can't read changehostname.c: No such file or directory
gcc: error: changehostname.c: No such file or directory
gcc: fatal error: no input files
compilation terminated.
gcc: error: changehostname.o: No such file or directory
ERROR: ld.so: object '/libchangehostname.so' from LD_PRELOAD cannot be preloaded (cannot open shared object file): ignored.
ERROR: ld.so: object '/libchangehostname.so' from LD_PRELOAD cannot be preloaded (cannot open shared object file): ignored.
2023-04-05 22:18:03,119 sagemaker-training-

UnexpectedStatusException: Error for Training job smile-uw-testing-2023-04-05-22-11-58-199: Failed. Reason: AlgorithmError: ExecuteUserScriptError:
ExitCode 1
ErrorMessage "│ ❱ 1223 │   │   │   │   │   raise ValueError("--tf32 requires Ampere or a new │
 │   1224 │   │   │   else:                                                     │
 │   1225 │   │   │   │   if is_torch_tf32_available():                         │
 │   1226 │   │   │   │   │   torch.backends.cuda.matmul.allow_tf32 = False     │
 ╰──────────────────────────────────────────────────────────────────────────────╯
 ValueError: --tf32 requires Ampere or a newer GPU arch, cuda>=11 and torch>=1.7"
Command "/opt/conda/bin/python3.9 BERTHA_AWS_Mortality.py --batch_size 64 --epochs 1 --lr 2.5e-05", exit code: 1